# 09 気候の比較

`01_data_fetch` で作った `data/processed/observations.parquet`（1 行 ＝ 都市 × 正時）から、比較ページ（`GET /compare`）の
グラフに使う月ごと・年ごとの値を集計し、`app/static/climate.json` に保存する（#25）。

- 集計の中身は `ml/climate.py` にある（テスト済み）。このノートブックは順に呼んで保存するだけ。
- 日ごとの集計は **現地の日付** で区切り、1 時間ごとの行がそろった日だけを使う（夏時間の切り替え日は 23・25 行）。
- 値は METAR（実測）を優先し、METAR にないもの（降水量・空全体の雲量）だけ Open-Meteo の過去データ（再解析）を使う。
- 降水を観測したかは、METAR の天気の記号で決める（WMO-No. 782 の Code table 4678。`docs/spec.md` 7 の表）。
- 既定は **サンプル**（`tests/fixtures/data_raw/` の 2 日ぶん。動くことの確認用で、数値に意味はない。一時的な場所に書く）。
  `app/static/climate.json` は、環境変数 `DATA_FETCH_MODE=full` を付けて全期間で実行したもの。外部とは通信しない。

出典：Weather data by Open-Meteo.com（CC BY 4.0）、Iowa Environmental Mesonet（Iowa State University）。


In [ ]:
import json
import os
import sys
import tempfile
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from app.cities import City
from ml import archive, climate, dataset, metar, raw_store

## 0. 設定・読み込み

In [ ]:
MODE = os.environ.get("DATA_FETCH_MODE", "sample")

if MODE == "full":
    OUTPUT = ROOT / "app" / "static" / "climate.json"
    YEARS = list(range(2018, 2026))
elif MODE == "sample":
    OUTPUT = Path(tempfile.mkdtemp()) / "climate.json"
    # fixture は 2025 年の 2 日ぶんだけ。
    YEARS = [2025]
else:
    raise ValueError("DATA_FETCH_MODE must be 'sample' or 'full'")

pd.set_option("display.float_format", "{:.2f}".format)
print(f"mode={MODE}", YEARS)

In [ ]:
if MODE == "full":
    observations = dataset.load_dataset(ROOT / "data" / "processed" / "observations.parquet")
else:
    raw_root = ROOT / "tests" / "fixtures" / "data_raw"
    parts = {}
    for city in City:
        weather_path = raw_store.raw_path(raw_root, "open_meteo_archive", city, 2025, "json")
        metar_path = raw_store.raw_path(raw_root, "iem", city, 2025, "csv")
        parts[city.value] = (
            archive.parse_archive(weather_path.read_bytes().decode("utf-8")),
            metar.parse_metar(metar_path.read_bytes().decode("utf-8")),
        )
    observations = dataset.build_dataset(parts)

dataset.validate_dataset(observations)
print(observations.shape, observations["time"].min(), observations["time"].max())

## 1. 集計と保存

値のない所（サンプルのデータにない月など）は `null`。NaN・無限大は書かない。

In [ ]:
summary = climate.climate_summary(observations, YEARS, MODE)

text = json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False) + "\n"
OUTPUT.write_text(text, encoding="utf-8", newline="\n")
print(OUTPUT.relative_to(ROOT) if MODE == "full" else "climate.json（一時的な場所）")

## 2. 中身の確認

In [ ]:
for city, values in summary["cities"].items():
    print(city)
    display(pd.DataFrame(values["monthly"]).set_index("month"))
    display(pd.DataFrame(values["yearly"]).set_index("year"))
    print("cloud_cover_share", values["cloud_cover_share"])